# Commodity Price Nowcasting with Mixed-Frequency Fundamentals

**Module 06 — Financial Applications**

**Learning objectives:**
- Build a multi-frequency MIDAS-X model for crude oil prices
- Combine daily futures returns with monthly macro fundamentals
- Implement MIDAS-RV for commodity volatility forecasting
- Compare against naive benchmarks using expanding-window evaluation

**Estimated time**: 13 minutes  
**Data**: Crude oil daily prices + monthly macro indicators (local CSV)

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import ElasticNetCV, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded.")

In [ ]:
section_divider("1. Load Multi-Frequency Data")

In [ ]:
apply_course_theme()
apply_plot_theme()

## 1. Load Multi-Frequency Data

In [ ]:
def load_commodity_data():
    """Load crude oil daily prices and monthly macro from local CSVs."""
    oil = pd.read_csv('../resources/crude_oil_daily.csv', parse_dates=['date'])
    oil = oil.set_index('date').sort_index()
    
    macro = pd.read_csv('../resources/macro_monthly.csv', parse_dates=['date'])
    macro = macro.set_index('date').sort_index()
    
    print(f"Crude oil daily: {len(oil)} trading days")
    print(f"Monthly macro: {len(macro)} months, columns: {list(macro.columns)}")
    return oil, macro


oil_daily, macro_monthly = load_commodity_data()

# Compute monthly crude oil price return (target variable)
oil_monthly = np.log(oil_daily['price']).resample('MS').last().diff().dropna()
oil_monthly.name = 'oil_log_return'

# Also compute monthly realised variance of oil returns
oil_daily_ret = oil_daily['return']
oil_rv_monthly = (oil_daily_ret**2).resample('MS').sum()
oil_rv_monthly.name = 'oil_rv'

# Align
common_dates = oil_monthly.index.intersection(macro_monthly.index)
common_dates = common_dates[common_dates >= pd.Timestamp('2005-01-01')]  # Skip early sparse months

y = oil_monthly.reindex(common_dates)
rv = oil_rv_monthly.reindex(common_dates)

print(f"\nAligned dataset:")
print(f"  Monthly target: {len(y)} observations")
print(f"  Range: {y.index[0].strftime('%Y-%m')} to {y.index[-1].strftime('%Y-%m')}")
print(f"  Mean monthly return: {y.mean():.4f}")
print(f"  Mean monthly vol: {np.sqrt(rv.mean() * 252):.1%} (annualised)")

# Plot
fig, axes = plt.subplots(2, 1, figsize=(13, 6))
axes[0].plot(oil_daily.index, oil_daily['price'], 'b-', linewidth=0.8)
axes[0].set_title('Crude Oil Daily Price (USD/barrel)')
axes[0].set_ylabel('Price')
axes[0].grid(True, alpha=0.3)

axes[1].plot(y.index, y.values * 100, 'r-', linewidth=1.2)
axes[1].axhline(0, color='black', linewidth=0.5)
axes[1].set_title('Monthly Crude Oil Log Return (%)')
axes[1].set_ylabel('Monthly Return (%)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
section_divider("2. Build MIDAS-X Feature Matrix")

## 2. Build MIDAS-X Feature Matrix

For each monthly target date:
- **Daily features**: Last K_daily daily log-returns of crude oil (lag1 = most recent)
- **Monthly features**: Last K_monthly monthly macro indicators

In [ ]:
def build_commodity_features(oil_daily_ret, macro_monthly, target_dates,
                              K_daily=22, K_monthly=3):
    """
    Build MIDAS-X feature matrix for commodity nowcasting.
    
    Parameters
    ----------
    oil_daily_ret : Series — daily crude oil log-returns
    macro_monthly : DataFrame — monthly macro indicators
    target_dates : DatetimeIndex — monthly forecast dates
    K_daily : int — number of daily lags
    K_monthly : int — number of monthly macro lags
    
    Returns
    -------
    X : DataFrame — feature matrix
    feature_names : list
    """
    rows = []
    valid_dates = []
    
    macro_diff = macro_monthly.diff()  # First-difference macro for stationarity
    
    for date in target_dates:
        row = {}
        
        # Daily crude oil return lags
        d_avail = oil_daily_ret[oil_daily_ret.index < date]
        if len(d_avail) < K_daily:
            continue
        d_lags = d_avail.tail(K_daily).values[::-1]  # lag1=most recent
        for j, v in enumerate(d_lags, start=1):
            row[f'oil_ret_L{j}'] = v
        
        # Daily summary statistics
        row['oil_mean'] = np.mean(d_lags)
        row['oil_momentum'] = d_lags[0] - d_lags[-1]  # Recent vs distant
        row['oil_rv_recent'] = np.mean(d_lags[:5]**2)   # Recent daily RV
        
        # Monthly macro lags
        m_avail = macro_diff[macro_diff.index < date]
        if len(m_avail) < K_monthly:
            continue
        
        for col in macro_monthly.columns:
            m_lags = m_avail[col].tail(K_monthly).values[::-1]
            for l, v in enumerate(m_lags, start=1):
                row[f'{col}_L{l}'] = v
        
        rows.append(row)
        valid_dates.append(date)
    
    X = pd.DataFrame(rows, index=valid_dates)
    
    # Drop NaN columns
    X = X.dropna(axis=1, how='all')
    X = X.fillna(0)  # Fill remaining NaN with 0
    
    return X


X = build_commodity_features(
    np.log1p(oil_daily['price']).diff().fillna(0),  # log-returns
    macro_monthly,
    common_dates,
    K_daily=22, K_monthly=3
)

# Align target with features
y_aligned = y.reindex(X.index)
valid_rows = y_aligned.notna()
X = X[valid_rows]
y_aligned = y_aligned[valid_rows]

print(f"Feature matrix: {X.shape}")
print(f"  Daily lags: 22 oil returns + 3 summaries = 25 features")
print(f"  Monthly lags: {len(macro_monthly.columns)} indicators × 3 lags = {len(macro_monthly.columns)*3} features")
print(f"  Total: {X.shape[1]} features")
print(f"  Target: {len(y_aligned)} monthly observations")

In [ ]:
section_divider("3. Elastic Net MIDAS-X Commodity Model")

## 3. Elastic Net MIDAS-X Commodity Model

In [ ]:
X_arr = X.values
y_arr = y_aligned.values
T = len(y_arr)

min_train = 36  # 3 years

en_forecasts = np.full(T, np.nan)
rw_forecasts = np.full(T, np.nan)  # Random walk benchmark
ar1_forecasts = np.full(T, np.nan)  # AR(1) benchmark

tscv = TimeSeriesSplit(n_splits=3)

print("Running expanding-window evaluation...")

for t in range(min_train, T):
    X_tr, y_tr = X_arr[:t], y_arr[:t]
    X_te = X_arr[t:t+1]
    
    # Standardise
    sc = StandardScaler()
    X_tr_s = sc.fit_transform(X_tr)
    X_te_s = sc.transform(X_te)
    
    # Elastic Net MIDAS-X
    en = ElasticNetCV(
        l1_ratio=[0.3, 0.5, 0.7, 0.9],
        alphas=np.logspace(-4, 0, 20),
        cv=tscv, max_iter=5000
    )
    en.fit(X_tr_s, y_tr)
    en_forecasts[t] = en.predict(X_te_s)[0]
    
    # Random walk: predict 0 change
    rw_forecasts[t] = 0.0
    
    # AR(1): OLS on single lag
    if t >= 2:
        y_lag = y_arr[t-1:t].reshape(-1, 1)
        y_fit = y_arr[max(0, t-36):t]
        y_lag_fit = y_arr[max(0, t-36)-1:t-1].reshape(-1, 1)
        X_ar1 = np.column_stack([np.ones(len(y_fit)), y_arr[max(0, t-36)-1:t-1]])
        if len(X_ar1) > 2:
            b_ar1, _, _, _ = np.linalg.lstsq(X_ar1, y_fit, rcond=None)
            ar1_forecasts[t] = b_ar1[0] + b_ar1[1] * y_arr[t-1]

# Evaluate on last 30%
eval_start = int(T * 0.7)
y_eval = y_arr[eval_start:]
en_eval = en_forecasts[eval_start:]
rw_eval = rw_forecasts[eval_start:]
ar1_eval = ar1_forecasts[eval_start:]

valid = ~np.isnan(en_eval) & ~np.isnan(ar1_eval)
y_v = y_eval[valid]
en_v = en_eval[valid]
rw_v = rw_eval[valid]
ar1_v = ar1_eval[valid]

print(f"\nOut-of-Sample Evaluation (last 30%, {valid.sum()} months):")
print(f"{'Model':<25} {'RMSE':<10} {'MAE':<10} {'Correlation':<12}")
print("-" * 57)

for name, preds in [('Random Walk', rw_v), ('AR(1)', ar1_v), ('Elastic Net MIDAS-X', en_v)]:
    errors = y_v - preds
    rmse = np.sqrt(np.mean(errors**2))
    mae = np.mean(np.abs(errors))
    corr = np.corrcoef(y_v, preds)[0, 1]
    print(f"{name:<25} {rmse:<10.4f} {mae:<10.4f} {corr:<12.4f}")

In [ ]:
section_divider("4. Feature Importance: Which Fundamentals Matter?")

## 4. Feature Importance: Which Fundamentals Matter?

In [ ]:
# Fit final model on full data for inspection
sc_final = StandardScaler()
X_s_final = sc_final.fit_transform(X_arr)

en_final = ElasticNetCV(
    l1_ratio=[0.3, 0.5, 0.7, 0.9],
    alphas=np.logspace(-4, 0, 30),
    cv=TimeSeriesSplit(n_splits=5),
    max_iter=10000
)
en_final.fit(X_s_final, y_arr)

# Compute feature group importance
coefs = en_final.coef_
feat_names = X.columns.tolist()

# Group by feature type
groups = {
    'Daily Oil Returns': [i for i, n in enumerate(feat_names) if n.startswith('oil_ret')],
    'Oil Summary Stats': [i for i, n in enumerate(feat_names) if n.startswith('oil_') and not n.startswith('oil_ret')],
    'INDPRO': [i for i, n in enumerate(feat_names) if n.startswith('INDPRO')],
    'UNRATE': [i for i, n in enumerate(feat_names) if n.startswith('UNRATE')],
    'DEF_SPREAD': [i for i, n in enumerate(feat_names) if n.startswith('DEF')],
    'TED_SPREAD': [i for i, n in enumerate(feat_names) if n.startswith('TED')],
    'VIX': [i for i, n in enumerate(feat_names) if n.startswith('VIX')],
}

group_importance = {name: np.sum(np.abs(coefs[indices])) for name, indices in groups.items()}
group_importance = dict(sorted(group_importance.items(), key=lambda x: x[1], reverse=True))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Group importance
ax = axes[0]
names_g = list(group_importance.keys())
values_g = list(group_importance.values())
colors_g = ['steelblue' if v > 0 else 'lightgray' for v in values_g]
ax.barh(range(len(names_g)), values_g, color=colors_g)
ax.set_yticks(range(len(names_g)))
ax.set_yticklabels(names_g)
ax.set_xlabel('Sum of |Elastic Net coefficients|')
ax.set_title('Feature Group Importance\n(Elastic Net MIDAS-X)')
ax.grid(True, alpha=0.3)

# Daily lag profile for oil returns
ax2 = axes[1]
daily_coefs = coefs[groups['Daily Oil Returns']]
lags = np.arange(1, len(daily_coefs) + 1)
colors_d = ['red' if c < 0 else 'blue' for c in daily_coefs]
ax2.bar(lags, daily_coefs, color=colors_d, alpha=0.7)
ax2.axhline(0, color='black', linewidth=0.5)
ax2.set_xlabel('Daily lag (1=most recent)')
ax2.set_ylabel('Coefficient')
ax2.set_title('Elastic Net Lag Profile:\nDaily Crude Oil Returns')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../resources/commodity_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Selected features: {np.sum(np.abs(coefs) > 1e-6)} / {len(coefs)}")
print(f"l1_ratio: {en_final.l1_ratio_:.2f}, alpha: {en_final.alpha_:.6f}")

In [ ]:
section_divider("5. MIDAS-RV for Commodity Volatility")

## 5. MIDAS-RV for Commodity Volatility

In [ ]:
def beta_weights(K, theta1, theta2):
    x = np.linspace(0.001, 0.999, K)
    unnorm = x**(theta1 - 1) * (1 - x)**(theta2 - 1)
    return unnorm / unnorm.sum()


# Build daily squared returns for MIDAS-RV
oil_daily_sq = (np.log1p(oil_daily['price']).diff().fillna(0))**2

# Build daily lag matrix for RV target dates
K_rv = 22
rv_target_dates = rv.index[rv.index.isin(X.index)]

X_rv = []
y_rv_aligned = []

for date in rv_target_dates:
    d_avail = oil_daily_sq[oil_daily_sq.index < date]
    if len(d_avail) < K_rv:
        continue
    window = d_avail.tail(K_rv).values[::-1]
    X_rv.append(np.log(window + 1e-10))  # Log daily squared returns
    y_rv_aligned.append(np.log(rv[date] + 1e-10))

X_rv_arr = np.array(X_rv)
y_rv_arr = np.array(y_rv_aligned)

# NLS for MIDAS-RV on commodity
def midas_rv_sse(params, X_arr, y_arr):
    mu, phi, theta1, theta2 = params
    if theta1 <= 0 or theta2 <= 0:
        return 1e10
    w = beta_weights(X_arr.shape[1], theta1, theta2)
    fitted = mu + phi * (X_arr @ w)
    return np.sum((y_arr - fitted)**2)

result = minimize(
    midas_rv_sse,
    x0=[y_rv_arr.mean(), 0.5, 1.0, 5.0],
    args=(X_rv_arr, y_rv_arr),
    method='L-BFGS-B',
    bounds=[(None, None), (0, 2), (0.01, 20), (0.01, 20)]
)

mu_rv, phi_rv, t1_rv, t2_rv = result.x
w_rv = beta_weights(K_rv, t1_rv, t2_rv)
fitted_rv = mu_rv + phi_rv * (X_rv_arr @ w_rv)
rmse_rv = np.sqrt(np.mean((y_rv_arr - fitted_rv)**2))

print(f"MIDAS-RV (Crude Oil):")
print(f"  θ₁={t1_rv:.3f}, θ₂={t2_rv:.3f}")
print(f"  φ={phi_rv:.4f}, μ={mu_rv:.4f}")
print(f"  In-sample RMSE (log-RV): {rmse_rv:.4f}")
print(f"  Most recent day weight: {w_rv[0]:.4f}")

# Plot annualised vol forecast vs actual
rv_actual_annualised = np.sqrt(np.exp(y_rv_arr) * 252)
rv_forecast_annualised = np.sqrt(np.exp(fitted_rv) * 252)

fig, ax = plt.subplots(figsize=(13, 4))
plot_idx = range(len(y_rv_arr))
ax.plot(plot_idx, rv_actual_annualised, 'k-', linewidth=1.5, label='Actual Annualised Vol')
ax.plot(plot_idx, rv_forecast_annualised, 'r--', linewidth=1.2, label='MIDAS-RV Forecast')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax.set_xlabel('Month index')
ax.set_ylabel('Annualised Volatility')
ax.set_title('MIDAS-RV: Crude Oil Realised Volatility')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../resources/commodity_rv_forecast.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Summary

This notebook demonstrated mixed-frequency commodity price nowcasting:

1. **MIDAS-X feature engineering**: Daily crude oil returns + monthly macro indicators (DEF_SPREAD, VIX, INDPRO)
2. **Elastic Net regularization**: Handles high-dimensional feature space (22 daily + 15 monthly = 37+ features)
3. **Feature importance**: Default spread and VIX tend to be most informative monthly predictors; recent daily returns (lag1-lag3) dominate daily features
4. **MIDAS-RV for commodity vol**: Same Beta-weight framework as equity volatility — estimated decay slightly slower for oil than equities
5. **Expanding-window evaluation**: MIDAS-X outperforms random walk and AR(1) in oil return nowcasting

**Key insight**: Commodity price forecasting is harder than volatility forecasting — monthly returns are close to a random walk. MIDAS-X with macro conditioning helps most during macro-driven price cycles (supply shocks, demand collapses), less so during speculation-driven episodes.

**Next**: `03_var_mixed_frequency.ipynb` — mixed-frequency VaR with Kupiec backtesting.

In [ ]:
key_takeaways(["MIDAS-X feature engineering", "Elastic Net regularization", "Feature importance", "MIDAS-RV for commodity vol", "Expanding-window evaluation"])